# DICOM QC - XNAT (with SSL/TLS Patch)

Batch QC review of DICOM data from an XNAT project.

**Use this notebook** when connecting to XNAT servers that require legacy TLS settings.

**Features:**
- TLS adapter for servers with legacy SSL configuration
- Save/load progress to avoid re-processing
- Interactive progress with ETA
- Incremental discovery (only add new scans)

**Setup:** Upload `dicom_qc.zip` to your workspace before running.

In [ ]:
import sys
import os
import shutil
import zipfile
from pathlib import Path

WORKSPACE = Path.cwd()

# Extract dicom_qc (overwrite if exists)
zip_path = WORKSPACE / 'dicom_qc.zip'
pkg_path = WORKSPACE / 'dicom_qc'
if zip_path.exists():
    if pkg_path.exists():
        shutil.rmtree(pkg_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(WORKSPACE)
    print('Extracted dicom_qc.zip')

sys.path.insert(0, str(WORKSPACE))

%matplotlib widget

In [ ]:
# TLS adapter for XNAT servers requiring legacy SSL settings
import ssl
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context

class TLSAdapter(HTTPAdapter):
    """Custom adapter for servers requiring legacy TLS settings."""
    def init_poolmanager(self, *args, **kwargs):
        ctx = create_urllib3_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        ctx.set_ciphers('DEFAULT:@SECLEVEL=1')
        try:
            ctx.options |= ssl.OP_LEGACY_SERVER_CONNECT
        except AttributeError:
            pass
        kwargs['ssl_context'] = ctx
        return super().init_poolmanager(*args, **kwargs)

# Patch requests.Session to auto-mount TLS adapter
_original_session_init = requests.Session.__init__

def _patched_session_init(self, *args, **kwargs):
    _original_session_init(self, *args, **kwargs)
    self.mount('https://', TLSAdapter())
    self.verify = False

requests.Session.__init__ = _patched_session_init
print('TLS adapter installed')

In [ ]:
# Connect to XNAT
import xnat

session = xnat.connect(
    server=os.environ['XNAT_HOST'],
    user=os.environ['XNAT_USER'],
    password=os.environ['XNAT_PASS'],
    verify=False
)
project = session.projects[os.environ['XNAT_PROJECT']]
print(f"Project: {project.name} ({len(project.subjects)} subjects)")

In [ ]:
from dicom_qc import QuickCheck

# Storage directory for this project
STORAGE_DIR = WORKSPACE / f'_dicom_qc_{os.environ["XNAT_PROJECT"]}'

# Initialize and load previous state if exists
qc = QuickCheck(data_dir=STORAGE_DIR)
qc.load_if_exists()
qc.connect_xnat(session)

In [ ]:
# Discover scans from XNAT (skips existing, adds new)
qc.discover_xnat(project, refresh=False)

In [ ]:
# Process series (run QC checks, generate thumbnails)
qc.process_all()

In [ ]:
# Generate HTML report
from IPython.display import FileLink, display

html_path, zip_path = qc.generate_html_report(WORKSPACE / f'qc_{os.environ["XNAT_PROJECT"]}_report.html')

# FileLink works in XNAT JupyterHub (report is in workspace)
display(FileLink(html_path.relative_to(WORKSPACE)))
if zip_path:
    print("For sharing:")
    display(FileLink(zip_path.relative_to(WORKSPACE)))

In [ ]:
# Interactive review
qc.display()

## Tips

### Start fresh
```python
qc.reset()               # Clears all data and storage
qc.discover_xnat(project)  # Re-scan XNAT project
```

### Re-process
```python
qc.process_all(reprocess=True)   # Re-run all
qc.process_all(retry_errors=True)  # Only retry errors
```